In [1]:
import pandas as pd
import numpy as np

from tqdm import tqdm

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity


# =========================================================
# LOAD DATA
# =========================================================

career_df = pd.read_csv(
    "csv/career_it_filtered.csv"
)

role_df = pd.read_csv(
    "csv/it_job_role.csv"
)

print("Career data :", len(career_df))
print("Role data   :", len(role_df))


# =========================================================
# CLEAN ROLE DATA
# =========================================================

role_df["Job Title"] = (
    role_df["Job Title"]
    .fillna("")
    .astype(str)
    .str.lower()
    .str.strip()
)

role_df["Skills"] = (
    role_df["Skills"]
    .fillna("")
    .astype(str)
    .str.lower()
    .str.strip()
)


# =========================================================
# ROLE TEXT
# =========================================================

role_df["role_text"] = (

    role_df["Job Title"] + " " +
    role_df["Skills"]

)


# =========================================================
# CLEAN CAREER DATA
# =========================================================

career_df["title"] = (
    career_df["title"]
    .fillna("")
    .astype(str)
    .str.lower()
    .str.strip()
)

career_df["body"] = (
    career_df["body"]
    .fillna("")
    .astype(str)
    .str.lower()
    .str.strip()
)


# =========================================================
# CAREER TEXT
# =========================================================

career_df["career_text"] = (

    career_df["title"] + " " +
    career_df["body"]

)


# =========================================================
# LOAD MODEL
# =========================================================

print("\nLoading model...")

model = SentenceTransformer(
    "sentence-transformers/all-mpnet-base-v2"
)


# =========================================================
# ENCODE ROLE DATA
# =========================================================

print("\nEncoding roles...")

role_embeddings = model.encode(

    role_df["role_text"].tolist(),

    batch_size=32,
    show_progress_bar=True,

    normalize_embeddings=True

)


# =========================================================
# ENCODE CAREER DATA
# =========================================================

print("\nEncoding career posts...")

career_embeddings = model.encode(

    career_df["career_text"].tolist(),

    batch_size=64,
    show_progress_bar=True,

    normalize_embeddings=True

)


# =========================================================
# MATCHING
# =========================================================

predicted_roles = []
role_scores = []

top3_roles = []
top3_scores = []

print("\nMatching roles...")

for career_emb in tqdm(career_embeddings):

    # =====================================================
    # COSINE SIMILARITY
    # =====================================================

    sims = cosine_similarity(
        [career_emb],
        role_embeddings
    )[0]


    # =====================================================
    # BEST ROLE
    # =====================================================

    best_idx = np.argmax(sims)

    best_role = role_df.iloc[best_idx][
        "Job Title"
    ]

    best_score = sims[best_idx]


    # =====================================================
    # TOP 3 ROLES
    # =====================================================

    top_indices = np.argsort(sims)[-3:][::-1]

    top_roles = [

        role_df.iloc[i]["Job Title"]

        for i in top_indices

    ]

    top_scores = [

        round(float(sims[i]), 4)

        for i in top_indices

    ]


    # =====================================================
    # THRESHOLD
    # =====================================================

    if best_score < 0.45:

        best_role = "unknown"


    # =====================================================
    # SAVE
    # =====================================================

    predicted_roles.append(best_role)

    role_scores.append(
        round(float(best_score), 4)
    )

    top3_roles.append(top_roles)

    top3_scores.append(top_scores)


# =========================================================
# SAVE RESULT
# =========================================================

career_df["predicted_role"] = predicted_roles

career_df["role_score"] = role_scores

career_df["top3_roles"] = top3_roles

career_df["top3_scores"] = top3_scores


# =========================================================
# SORT BY SCORE
# =========================================================

career_df = career_df.sort_values(
    by="role_score",
    ascending=False
)


# =========================================================
# PREVIEW
# =========================================================

preview_cols = [

    "title",
    "predicted_role",
    "role_score",
    "top3_roles",
    "top3_scores"

]

print("\nTOP RESULTS:\n")

print(
    career_df[preview_cols]
    .head(20)
)


# =========================================================
# SAVE CSV
# =========================================================

output_path = "career_role_prediction.csv"

career_df.to_csv(

    output_path,

    index=False,
    encoding="utf-8-sig"

)

print("\nDONE!")
print("Saved to:", output_path)

C:\Users\Gregorius Christian\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Career data : 2942
Role data   : 289

Loading model...

Encoding roles...


Batches: 100%|██████████| 10/10 [00:07<00:00,  1.32it/s]



Encoding career posts...


Batches: 100%|██████████| 46/46 [08:52<00:00, 11.57s/it]



Matching roles...


100%|██████████| 2942/2942 [00:07<00:00, 371.85it/s]



TOP RESULTS:

                                                  title  \
2360       what are the skills required for vfx artist?   
75    web developers out there can you provide guida...   
757   interested in developing a career in the busin...   
66    what are possible career s paths for data busi...   
1045  what are the best opportunities for a manageme...   
187     what should i learn for mobile app development?   
70    what are possible career paths for a data busi...   
1665                    what is the demand for powerbi?   
286   please help me out to build my career as a web...   
10    data analysts what are the most important exce...   
683   how can data analytics skills be utilized for ...   
1455  subject combination for someone going for soft...   
180   what do i need to learn to transition to a car...   
1116      jobs for data driven and analytical thinkers?   
1161  how can i learn what being a business intellig...   
391   what job would let me get data from

In [2]:
df_career_role = pd.read_csv("career_role_prediction.csv")

In [3]:
df_career_role.head()

,title,body,dominant_topic_name,url,good_score,bad_score,keyword_match,negative_match,final_score,career_text,predicted_role,role_score,top3_roles,top3_scores
0,what are the skills required for vfx artist?,passionate about vfx. wanted to know more abou...,Career Change for Non-Corporate Artist,https://www.reddit.com/r/careerguidance/commen...,0.3588,0.1249,False,False,0.3307,what are the skills required for vfx artist? p...,vfx artist,0.7777,"['vfx artist', 'visual effects animator', 'gra...","[0.7777, 0.6448, 0.6029]"
1,web developers out there can you provide guida...,i am a 4th year b.tech student (electronics an...,Software Development Career Transition,https://www.reddit.com/r/careerguidance/commen...,0.5111,0.2009,True,False,0.5326,web developers out there can you provide guida...,entry level web developer,0.7579,"['entry level web developer', 'web developer',...","[0.7579, 0.7473, 0.7355]"
2,interested in developing a career in the busin...,hello! i m a business administration and i m r...,Career Advancement in Data Analytics,https://www.reddit.com/r/careerguidance/commen...,0.5177,0.3283,True,False,0.4385,interested in developing a career in the busin...,business intelligence analyst,0.7509,"['business intelligence analyst', 'power bi an...","[0.7509, 0.6451, 0.589]"
3,what are possible career s paths for data busi...,hi i am working in a company as data analyst d...,Career Advancement in Data Analytics,https://www.reddit.com/r/careerguidance/commen...,0.6199,0.3609,True,False,0.5352,what are possible career s paths for data busi...,business intelligence analyst,0.7503,"['business intelligence analyst', 'power bi an...","[0.7503, 0.6684, 0.6515]"
4,what are the best opportunities for a manageme...,hi r careerguidance! i am currently a manageme...,Career Advancement in Data Analytics,http://www.reddit.com/r/careerguidance/comment...,0.4888,0.3160,True,False,0.4138,what are the best opportunities for a manageme...,business intelligence analyst,0.7425,"['business intelligence analyst', 'power bi an...","[0.7425, 0.6556, 0.5415]"


In [6]:
df_career_role = df_career_role[['title', 'body', 'dominant_topic_name', 'career_text', 'predicted_role', 'top3_roles']]

In [7]:
df_career_role.head(20)

,title,body,dominant_topic_name,career_text,predicted_role,top3_roles
0,what are the skills required for vfx artist?,passionate about vfx. wanted to know more abou...,Career Change for Non-Corporate Artist,what are the skills required for vfx artist? p...,vfx artist,"['vfx artist', 'visual effects animator', 'gra..."
1,web developers out there can you provide guida...,i am a 4th year b.tech student (electronics an...,Software Development Career Transition,web developers out there can you provide guida...,entry level web developer,"['entry level web developer', 'web developer',..."
2,interested in developing a career in the busin...,hello! i m a business administration and i m r...,Career Advancement in Data Analytics,interested in developing a career in the busin...,business intelligence analyst,"['business intelligence analyst', 'power bi an..."
3,what are possible career s paths for data busi...,hi i am working in a company as data analyst d...,Career Advancement in Data Analytics,what are possible career s paths for data busi...,business intelligence analyst,"['business intelligence analyst', 'power bi an..."
4,what are the best opportunities for a manageme...,hi r careerguidance! i am currently a manageme...,Career Advancement in Data Analytics,what are the best opportunities for a manageme...,business intelligence analyst,"['business intelligence analyst', 'power bi an..."
5,what should i learn for mobile app development?,my goal is eventually to master cross platform...,Software Development Career Transition,what should i learn for mobile app development...,mobile developer,"['mobile developer', 'mobile app developer', '..."
6,what are possible career paths for a data busi...,hi i am working in a company as data analyst d...,Career Advancement in Data Analytics,what are possible career paths for a data busi...,business intelligence analyst,"['business intelligence analyst', 'power bi an..."
7,what is the demand for powerbi?,what jobs demand knowledge of powerbi and sql?,Career Advancement in Data Analytics,what is the demand for powerbi? what jobs dema...,power bi analyst,"['power bi analyst', 'power bi developer', 'bu..."
8,please help me out to build my career as a web...,i want to learn following things html5 css3 re...,Software Development Career Transition,please help me out to build my career as a web...,entry level web developer,"['entry level web developer', 'web developer',..."
9,data analysts what are the most important exce...,i am planning to take courses through coursera...,Career Advancement in Data Analytics,data analysts what are the most important exce...,data analyst,"['data analyst', 'business intelligence analys..."
